In [2]:
import polars, sys; print(sys.executable, polars.__version__)


/Users/danielsanchezcuenca/Desktop/GCP/redelectrica-proyecto/.venv/bin/python 1.32.3


In [2]:
import polars as pl
import json
import os
from __future__ import annotations
import json
from typing import Any, Dict, List

In [3]:
## función para leer json en local

def leer_json(path:str) -> Dict[str, Any]:

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)
    
    


In [4]:
json = leer_json("/Users/danielsanchezcuenca/Desktop/GCP/redelectrica-proyecto/raw/estructura.json")

In [5]:
print(json)

{'requests': {'start_date': '2018-01-01T00:00', 'end_date': '2018-12-31T23:59', 'time_trunc': 'year', 'geo_trunc': 'electric_system', 'geo_limit': 'peninsular', 'geo_ids': '8741'}, 'data': {'data': {'type': 'Generación por tecnología', 'id': 'gen1', 'attributes': {'title': 'Generación por tecnología', 'last-update': '2019-06-12T17:40:36.000+02:00', 'description': None}, 'meta': {'cache-control': {'cache': 'HIT', 'expireAt': '2025-09-28T12:20:01'}}}, 'included': [{'type': 'Hidráulica', 'id': '10288', 'groupId': '1', 'attributes': {'title': 'Hidráulica', 'description': None, 'color': '#0090d1', 'icon': None, 'type': 'Renovable', 'magnitude': None, 'composite': False, 'last-update': '2019-06-12T17:40:36.000+02:00', 'values': [{'value': 34113964.237, 'percentage': 0.139296443828509, 'datetime': '2018-01-01T00:00:00.000+01:00'}]}}, {'type': 'Nuclear', 'id': '1446', 'groupId': '1', 'attributes': {'title': 'Nuclear', 'description': None, 'color': '#464394', 'icon': None, 'type': 'No-Renovable

In [15]:
import polars as pl
from typing import Any, Dict, List

def aplanar_estructura(raw: Dict[str, Any]) -> pl.DataFrame:
    rows: List[Dict[str, Any]] = []

    # --- Categoría padre (p. ej. "Generación por tecnología") ---
    parent = raw.get("data", {}).get("data", {})  # en tu JSON es un dict
    parent_attrs = parent.get("attributes", {}) if isinstance(parent, dict) else {}
    parent_category = parent_attrs.get("title") or parent.get("type") or "desconocido"
    parent_id = parent.get("id")

    # --- Función recursiva por si existen hijos en 'attributes.content' ---
    def visita_nodo(nodo: Dict[str, Any]):
        attrs = nodo.get("attributes", {})
        sub_type = attrs.get("title") or nodo.get("type")        # p.ej. "Hidráulica"
        energy_type = attrs.get("type")                           # p.ej. "Renovable"/"No-Renovable"

        # Caso hoja: values
        for v in attrs.get("values", []):
            rows.append({
                "dt": v.get("datetime"),
                "value": v.get("value"),
                "percentage": v.get("percentage"),
                "energy_type": energy_type,
                "sub_type": sub_type,
                "parent_category": parent_category,  # << padre
                "parent_id": parent_id,
                "source": "ree/balance",
            })

        # Caso intermedio: hijos en content
        for hijo in attrs.get("content", []):
            visita_nodo(hijo)

    # --- Recorremos los nodos principales ---
    for nodo in raw.get("data", {}).get("included", []):
        visita_nodo(nodo)

    # --- DataFrame tipado ---
    df = pl.DataFrame(rows).with_columns(
        pl.col("dt").str.strptime(pl.Datetime, strict=False),
        pl.col("value").cast(pl.Float64, strict=False),
        pl.col("percentage").cast(pl.Float64, strict=False),
        pl.col("energy_type").cast(pl.String),
        pl.col("sub_type").cast(pl.String),
        pl.col("parent_category").cast(pl.String),
        pl.col("parent_id").cast(pl.String, strict=False),
        pl.col("source").cast(pl.String),
    )
    return df


In [16]:
df = aplanar_estructura(json)
df.head()

dt,value,percentage,energy_type,sub_type,parent_category,parent_id,source
"datetime[μs, UTC]",f64,f64,str,str,str,str,str
2017-12-31 23:00:00 UTC,3.4114e7,0.139296,"""Renovable""","""Hidráulica""","""Generación por tecnología""","""gen1""","""ree/balance"""
2017-12-31 23:00:00 UTC,5.3198e7,0.21722,"""No-Renovable""","""Nuclear""","""Generación por tecnología""","""gen1""","""ree/balance"""
2017-12-31 23:00:00 UTC,3.4881e7,0.142429,"""No-Renovable""","""Carbón""","""Generación por tecnología""","""gen1""","""ree/balance"""
2017-12-31 23:00:00 UTC,-0.001,4.0833e-12,"""No-Renovable""","""Fuel + Gas""","""Generación por tecnología""","""gen1""","""ree/balance"""
2017-12-31 23:00:00 UTC,2.6403e7,0.10781,"""No-Renovable""","""Ciclo combinado""","""Generación por tecnología""","""gen1""","""ree/balance"""


In [17]:
if df.is_empty():
    print("balance está vacío")

    

In [18]:
df_unico = df.unique(subset=["dt","energy_type","sub_type","parent_category"])

In [19]:
df_clean = df_unico.filter(
    ~ pl.all_horizontal(pl.all().is_null())
)

In [20]:
df_clean.head()

dt,value,percentage,energy_type,sub_type,parent_category,parent_id,source
"datetime[μs, UTC]",f64,f64,str,str,str,str,str
2017-12-31 23:00:00 UTC,2.6403e7,0.10781,"""No-Renovable""","""Ciclo combinado""","""Generación por tecnología""","""gen1""","""ree/balance"""
2017-12-31 23:00:00 UTC,-0.001,4.0833e-12,"""No-Renovable""","""Fuel + Gas""","""Generación por tecnología""","""gen1""","""ree/balance"""
2017-12-31 23:00:00 UTC,3.4881e7,0.142429,"""No-Renovable""","""Carbón""","""Generación por tecnología""","""gen1""","""ree/balance"""
2017-12-31 23:00:00 UTC,4.4243e6,0.018066,"""Renovable""","""Solar térmica""","""Generación por tecnología""","""gen1""","""ree/balance"""
2017-12-31 23:00:00 UTC,7.3805e6,0.030137,"""Renovable""","""Solar fotovoltaica""","""Generación por tecnología""","""gen1""","""ree/balance"""


In [28]:
def normalize_nulls(df: pl.DataFrame) -> pl.DataFrame:

    out = df

    for col,dtype in df.schema.items():
        if dtype in(pl.Float64, pl.Float32):
            out = out.with_columns(pl.col(col).fill_nan(None))
        if dtype in (pl.String,pl.Utf8):
            out = out.with_columns(pl.when(pl.col(col).str.strip_chars()=="")
                                   .then(None)
                                   .otherwise(pl.col(col))
                                   .alias(col)
                                   )
    return out

In [29]:
df_clean = normalize_nulls(df)

In [32]:
def star_date(raw: Dict[str, Any]):
    fecha = raw.get("requests", {}).get("start_date")
    # Castear fecha a formato YYYY-MM-DD
    if fecha:
        fecha = fecha[:10]
    return fecha

In [ ]:
fecha = star_date(json)
print(fecha)


2018-01-01
